# BPfold

In [ ]:
import os
import pandas as pd
import time
import subprocess
from pathlib import Path

In [ ]:
method_name = "BPfold"
base = Path.cwd()
bpfold_dir = base.parent / "tools" / "BPfold"
checkpoint_dir = bpfold_dir / "model_predict"
prediction_dir = base.parent / "prediction"

print(f"Base directory: {base}")
print(f"BPfold directory: {bpfold_dir}")
print(f"Checkpoint directory: {checkpoint_dir}")
print(f"Prediction directory: {prediction_dir}")

In [ ]:
# Download and extract model files if they don't exist
if not checkpoint_dir.exists():
    print(f"Model directory {checkpoint_dir} not found! Downloading...")
    bpfold_dir.mkdir(parents=True, exist_ok=True)
    
    # Download model
    download_url = "https://github.com/heqin-zhu/BPfold/releases/latest/download/model_predict.tar.gz"
    tar_file = bpfold_dir / "model_predict.tar.gz"
    
    print(f"Downloading from {download_url}...")
    wget_cmd = f"wget -O {tar_file} {download_url}"
    result = subprocess.run(wget_cmd, shell=True, capture_output=True, text=True)
    
    if result.returncode != 0:
        print(f"Error downloading: {result.stderr}")
    else:
        print(f"Downloaded successfully to {tar_file}")
        # Extract the tar file
        extract_cmd = f"tar -xzf {tar_file} -C {bpfold_dir}"
        result = subprocess.run(extract_cmd, shell=True, capture_output=True, text=True)
        if result.returncode != 0:
            print(f"Error extracting: {result.stderr}")
        else:
            print(f"Extracted successfully to {checkpoint_dir}")
            # Remove tar file
            tar_file.unlink()
            print(f"Cleaned up tar file")
else:
    print(f"Model directory {checkpoint_dir} already exists")

# Check if BPfold is installed and model files exist
import sys
sys.path.append(str(bpfold_dir / "src"))

# Install BPfold if not already installed
try:
    import BPfold
    print("BPfold is already installed")
except ImportError:
    print("Installing BPfold...")
    subprocess.run(["pip", "install", "BPfold"], capture_output=True, text=True)
    try:
        import BPfold
        print("BPfold installed successfully")
    except ImportError:
        print("Failed to install BPfold!")

# Check if model files exist
if checkpoint_dir.exists():
    model_files = list(checkpoint_dir.glob("*.pth"))
    print(f"Found {len(model_files)} model files in {checkpoint_dir}")
else:
    print(f"Warning: Model directory {checkpoint_dir} not found!")

In [ ]:
def read_virus_fasta(path: str):
    lines = [ln.strip() for ln in open(path, 'r').read().splitlines() if ln.strip() != '']
    records = []
    for i in range(0, len(lines), 3):
        header, seq, struct = lines[i], lines[i+1], lines[i+2]
        name = header[1:].strip()
        records.append((name, seq.strip(), struct.strip()))
    df = pd.DataFrame(records, columns=['name','sequence','structure']).set_index('name')
    return df

viruses = read_virus_fasta('../data/viruses.fasta')

selected_virus_keys = None

if selected_virus_keys is None:
    virus_ids = list(viruses.index)
else:
    tmp = []
    for k in selected_virus_keys:
        if isinstance(k, int):
            tmp.append(viruses.index[k])
        else:
            tmp.append(str(k))
    virus_ids = tmp

# Compute structures using BPfold

In [ ]:
def run_bpfold_prediction(input_fasta, output_dir="BPfold_temp_results"):
    """Run BPfold prediction on a fasta file"""
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Run BPfold command
    cmd = [
        "BPfold",
        "--checkpoint_dir", str(checkpoint_dir),
        "--input", input_fasta,
        "--output", output_dir,
        "--out_type", "dbn",
        "--gpu", "2"
    ]
    
    subprocess.run(cmd, capture_output=True, text=True)
    
    return output_dir

In [ ]:
# Create prediction directory if it doesn't exist
prediction_dir.mkdir(parents=True, exist_ok=True)

out_fasta_name = method_name
output_fasta_path = prediction_dir / (out_fasta_name + ".fasta")

# Remove existing output file if it exists
if output_fasta_path.exists():
    output_fasta_path.unlink()

print(f"Results will be saved to {output_fasta_path}")
print(f"{' ':3}\t{'virus':<20}\t{'len':<5}\t{'time'}")

for i, vid in enumerate(virus_ids):
    start_time = time.time()
    seq = viruses.loc[vid]['sequence']
    print(f"{i+1:3d}/{len(virus_ids)}\t{vid:<20}\t{len(seq):<5}\t", end='', flush=True)

    # Write a one-sequence fasta file
    input_fasta = f"BPfold_input_{i}.fasta"
    with open(input_fasta, "w") as ofile:
        ofile.write(f">{vid}\n{seq}\n")

    # Run BPfold prediction
    output_dir = run_bpfold_prediction(input_fasta, f"BPfold_temp_{i}")
    elapsed_time = time.time() - start_time

    # Find and read the output file
    dbn_files = list(Path(output_dir).glob("*.dbn"))
    if dbn_files:
        dbn_file = dbn_files[0]
        with open(dbn_file, 'r') as f:
            lines = f.readlines()
            predicted_seq = lines[0].strip()
            structure = lines[1].strip()
            
            # Write three lines to output file in prediction directory
            with open(output_fasta_path, "a") as out_f:
                out_f.write(f">{vid}\n")
                out_f.write(f"{predicted_seq}\n")
                out_f.write(f"{structure}\n")

    print(f"{elapsed_time: .1f} s")

    os.remove(input_fasta)
    import shutil
    shutil.rmtree(output_dir)

print(f"\nPredictions completed! Results saved to {output_fasta_path}")